# 🌸 Genshin Impact Hoyolab Wiki AI Agent

This notebook lets you:
1. **Scrape** Hoyolab Genshin Impact wiki pages and save them as local `.txt` files
2. **Ask DeepSeek** questions — it reads the saved files to answer you

---
## Setup

Install required libraries:

In [ ]:
# pip install google-generativeai requests beautifulsoup4 playwright nest_asyncio
# pip install playwright
# playwright install chromium
# pip install ipywidgets

In [1]:
import os

# --- CONFIGURE THESE ---
WIKI_DATA_DIR  = "./wiki_data"                 # Folder where scraped .txt files are saved
# -----------------------

os.makedirs(WIKI_DATA_DIR, exist_ok=True)
print(f"✅ Wiki data will be saved to: {os.path.abspath(WIKI_DATA_DIR)}")

✅ Wiki data will be saved to: C:\Users\Anisa\OneDrive\Documents\Genshin AI Agent\wiki_data


In [2]:
import asyncio
import sys

# Force the Proactor event loop for Windows; fix Playwright asyncio issue on Windows
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())


## 2. Wiki Scraper

Scrapes a Hoyolab Genshin Impact wiki page and saves the text locally.

**How to find wiki URLs:**
- Go to https://wiki.hoyolab.com/pc/genshin/home
- Navigate to any character, weapon, or game mechanic page
- Copy that URL and paste it below

In [3]:
from playwright.sync_api import sync_playwright
from pathlib import Path
import re
import time

def sanitize_filename(name: str) -> str:
    """Turn a page title or URL slug into a safe filename."""
    name = re.sub(r'[^\w\s-]', '', name).strip()
    name = re.sub(r'[\s-]+', '_', name)
    return name[:80]


def _scrape_wiki_page(url: str, label: str = None) -> str:
    """
    Scrapes a Hoyolab wiki page using a headless browser.
    Uses the sync Playwright API to avoid asyncio issues on Windows + Python 3.13.
    """
    print(f"🌐 Opening: {url}")

    with sync_playwright() as p:
        browser = p.chromium.launch(channel="chrome",headless=True)
        page = browser.new_page(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                       "AppleWebKit/537.36 (KHTML, like Gecko) "
                       "Chrome/120.0.0.0 Safari/537.36"
        )

        page.goto(url, wait_until="domcontentloaded", timeout=30000)

        # Wait longer for Hoyolab's JS to fully render
        page.wait_for_timeout(6000)

        # Wait specifically for Hoyolab wiki content
        try:
            page.wait_for_selector(
                ".wiki-entry-content, .entry-detail, .wiki-detail, "
                ".obc-tmpl, [class*='wiki'], [class*='entry'], [class*='detail']",
                timeout=20000
            )
            print("  ✅ Wiki content detected.")
        except Exception:
            print("  ⚠️  Could not detect wiki content — scraping whatever is loaded.")

        # Scroll down to trigger any lazy-loaded content
        page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
        page.wait_for_timeout(2000)  # Wait for lazy content to load after scroll


        # Get page title
        title = page.title()
        title = title.replace(" - HoYoLAB", "").replace(" | HoYoLAB", "").strip()

        # Extract visible text, removing clutter
        content = page.evaluate("""
            () => {
                const remove = ['nav', 'footer', 'script', 'style',
                                'header', 'aside', '[class*="ad"]',
                                '[class*="banner"]', '[class*="sidebar"]'];
                remove.forEach(sel => {
                    document.querySelectorAll(sel).forEach(el => el.remove());
                });
                return document.body.innerText;
            }
        """)

        browser.close()

    # Clean up whitespace
    lines = [line.strip() for line in content.splitlines()]
    lines = [l for l in lines if l]
    clean_text = "\n".join(lines)

    # Determine filename
    file_label = label if label else sanitize_filename(title if title else url.split("/")[-1])
    file_path = Path(WIKI_DATA_DIR) / f"{file_label}.txt"

    # Save with metadata header
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(f"SOURCE URL: {url}\n")
        f.write(f"PAGE TITLE: {title}\n")
        f.write(f"{'='*60}\n\n")
        f.write(clean_text)

    print(f"✅ Saved: {file_path}  ({len(clean_text):,} characters)")
    return str(file_path)


import threading

def scrape_wiki_page(url: str, label: str = None) -> str:
    """
    Runs the sync Playwright scraper in a separate thread to avoid
    Jupyter's background asyncio loop blocking it.
    """
    result = [None]
    error  = [None]

    def run():
        try:
            result[0] = _scrape_wiki_page(url, label)
        except Exception as e:
            error[0] = e

    thread = threading.Thread(target=run)
    thread.start()
    thread.join()

    if error[0]:
        raise error[0]
    return result[0]


print("✅ Scraper ready.")


✅ Scraper ready.


In [4]:
# """
# hoyowiki_scraper.py
# -------------------
# Scrapes Hoyolab wiki pages (Genshin Impact) and saves complete text to disk.

# Key improvements over v1:
#   - Waits for network idle so JS-rendered content is fully settled
#   - Clicks every visible tab and collects content per-tab
#   - Expands all accordion / collapsible sections before extraction
#   - Incremental scroll with pauses so lazy-loaded images / tables appear
#   - Targeted extraction from the actual wiki content container
#     (falls back to full body if the selector isn't found)
#   - Preserves table structure as readable plain-text
#   - Does NOT nuke sidebar/nav until after tab discovery, so tab bars survive
#   - Runs Playwright in its own thread to stay Jupyter-safe
# """

# from playwright.sync_api import sync_playwright, TimeoutError as PWTimeout
# from pathlib import Path
# import re
# import threading

# # ── Config ────────────────────────────────────────────────────────────────────
# WIKI_DATA_DIR = "wiki_data"
# Path(WIKI_DATA_DIR).mkdir(parents=True, exist_ok=True)

# # Selectors that wrap all meaningful wiki content on Hoyolab
# CONTENT_SELECTORS = [
#     ".wiki-entry-content",
#     ".entry-detail",
#     ".wiki-detail",
#     ".obc-tmpl-part",
#     "[class*='wiki-entry']",
#     "[class*='entry-content']",
#     "[class*='obc-tmpl']",
# ]

# # Tab bar selectors (Hoyolab uses several patterns across page types)
# TAB_SELECTORS = [
#     ".obc-tab-item",
#     ".wiki-tab-item",
#     "[class*='tab-item']",
#     "[class*='tab__item']",
#     "[role='tab']",
# ]

# # Accordion / "show more" selectors
# EXPAND_SELECTORS = [
#     "[class*='collapse-btn']",
#     "[class*='expand-btn']",
#     "[class*='show-more']",
#     "[class*='toggle-btn']",
#     "details:not([open]) > summary",
#     "[aria-expanded='false']",
# ]

# # Elements to strip AFTER tab/content extraction
# STRIP_SELECTORS = [
#     "nav", "header", "footer", "script", "style",
#     "aside", "[class*='sidebar']", "[class*='banner']",
#     "[class*=' ad-']", "[id*='ad-']",
# ]

# # ── Helpers ───────────────────────────────────────────────────────────────────

# def sanitize_filename(name: str) -> str:
#     name = re.sub(r'[^\w\s-]', '', name).strip()
#     name = re.sub(r'[\s-]+', '_', name)
#     return name[:80]


# def _table_to_text(table_el) -> str:
#     """
#     Convert a Playwright ElementHandle pointing at a <table> into readable text.
#     Each row becomes one line; cells are pipe-separated.
#     """
#     rows = table_el.query_selector_all("tr")
#     lines = []
#     for row in rows:
#         cells = row.query_selector_all("th, td")
#         cell_texts = [c.inner_text().strip().replace("\n", " ") for c in cells]
#         if any(cell_texts):
#             lines.append(" | ".join(cell_texts))
#     return "\n".join(lines)


# def _slow_scroll(page, steps: int = 12, delay_ms: int = 600) -> None:
#     """Scroll the page incrementally so lazy loaders have time to fire."""
#     for i in range(1, steps + 1):
#         page.evaluate(f"window.scrollTo(0, document.body.scrollHeight * {i}/{steps})")
#         page.wait_for_timeout(delay_ms)
#     # Scroll back to top then bottom once more to catch anything missed
#     page.evaluate("window.scrollTo(0, 0)")
#     page.wait_for_timeout(300)
#     page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
#     page.wait_for_timeout(800)


# def _expand_all(page) -> int:
#     """Click every collapsed accordion / toggle found on the page. Returns count clicked."""
#     clicked = 0
#     for sel in EXPAND_SELECTORS:
#         try:
#             elements = page.query_selector_all(sel)
#             for el in elements:
#                 try:
#                     if el.is_visible():
#                         el.click()
#                         page.wait_for_timeout(200)
#                         clicked += 1
#                 except Exception:
#                     pass
#         except Exception:
#             pass
#     return clicked


# def _extract_content_text(page) -> str:
#     """
#     Try to grab text from the focused wiki content container.
#     Falls back to full body text if the targeted selectors aren't found.
#     Tables are converted to pipe-delimited rows before text extraction.
#     """
#     # Convert tables to readable text in-place before innerText strips structure
#     tables = page.query_selector_all("table")
#     table_texts = {}
#     for idx, tbl in enumerate(tables):
#         try:
#             marker = f"__TABLE_{idx}__"
#             table_texts[marker] = _table_to_text(tbl)
#             page.evaluate(
#                 f"""(tbl) => {{
#                     const ph = document.createElement('pre');
#                     ph.textContent = '{marker}';
#                     tbl.parentNode.replaceChild(ph, tbl);
#                 }}""",
#                 tbl,
#             )
#         except Exception:
#             pass

#     # Strip noise elements
#     for sel in STRIP_SELECTORS:
#         try:
#             page.evaluate(
#                 f"""document.querySelectorAll('{sel}')
#                     .forEach(el => el.remove())"""
#             )
#         except Exception:
#             pass

#     # Try targeted content containers first
#     raw = ""
#     for sel in CONTENT_SELECTORS:
#         try:
#             el = page.query_selector(sel)
#             if el:
#                 raw = el.inner_text()
#                 if len(raw.strip()) > 200:          # looks like real content
#                     break
#         except Exception:
#             pass

#     # Full-body fallback
#     if len(raw.strip()) < 200:
#         raw = page.evaluate("() => document.body.innerText")

#     # Restore table text
#     for marker, table_text in table_texts.items():
#         raw = raw.replace(marker, f"\n{table_text}\n")

#     # Clean whitespace
#     lines = [line.strip() for line in raw.splitlines()]
#     lines = [l for l in lines if l]
#     return "\n".join(lines)


# # ── Core scraper ──────────────────────────────────────────────────────────────

# def _scrape_wiki_page(url: str, label: str = None) -> str:
#     print(f"\n🌐 Opening: {url}")

#     with sync_playwright() as p:
#         browser = p.chromium.launch(
#             channel="chrome",
#             headless=True,
#             args=["--disable-blink-features=AutomationControlled"],
#         )
#         context = browser.new_context(
#             user_agent=(
#                 "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
#                 "AppleWebKit/537.36 (KHTML, like Gecko) "
#                 "Chrome/124.0.0.0 Safari/537.36"
#             ),
#             viewport={"width": 1440, "height": 900},
#         )
#         page = context.new_page()

#         # ── 1. Navigate and wait for JS to settle ─────────────────────────
#         page.goto(url, wait_until="domcontentloaded", timeout=40000)

#         # Give the initial JS bundle time to boot
#         page.wait_for_timeout(4000)

#         # Then wait for network to go quiet (images, XHR data fetches)
#         try:
#             page.wait_for_load_state("networkidle", timeout=20000)
#         except PWTimeout:
#             print("  ⚠️  networkidle timeout — continuing anyway.")

#         # Wait for at least one content selector to appear
#         try:
#             page.wait_for_selector(
#                 ", ".join(CONTENT_SELECTORS + ["[class*='wiki']", "[class*='obc']"]),
#                 timeout=15000,
#             )
#             print("  ✅ Wiki content detected.")
#         except PWTimeout:
#             print("  ⚠️  Content selector not found — will scrape full page.")

#         # ── 2. Initial scroll to trigger lazy loads ────────────────────────
#         _slow_scroll(page, steps=10, delay_ms=500)

#         # ── 3. Discover and iterate over all tabs ─────────────────────────
#         tab_selector = ", ".join(TAB_SELECTORS)
#         tabs = page.query_selector_all(tab_selector)

#         page_title = page.title()
#         page_title = (
#             page_title.replace(" - HoYoLAB", "")
#                       .replace(" | HoYoLAB", "")
#                       .strip()
#         )

#         sections: list[tuple[str, str]] = []   # (tab_name, content_text)

#         if tabs:
#             print(f"  📑 Found {len(tabs)} tab(s) — iterating each one.")
#             for idx, tab in enumerate(tabs):
#                 tab_name = ""
#                 try:
#                     tab_name = tab.inner_text().strip() or f"Tab_{idx+1}"
#                     tab.scroll_into_view_if_needed()
#                     tab.click()
#                     # Wait for the new panel to render
#                     page.wait_for_timeout(2500)
#                     try:
#                         page.wait_for_load_state("networkidle", timeout=8000)
#                     except PWTimeout:
#                         pass
#                     # Scroll and expand within the newly visible panel
#                     _slow_scroll(page, steps=8, delay_ms=400)
#                     expanded = _expand_all(page)
#                     if expanded:
#                         page.wait_for_timeout(600)
#                     text = _extract_content_text(page)
#                     sections.append((tab_name, text))
#                     print(f"    ✔ Tab '{tab_name}': {len(text):,} chars")
#                 except Exception as e:
#                     print(f"    ⚠️  Tab '{tab_name}' error: {e}")
#         else:
#             # No tabs — single-panel page
#             print("  📄 No tabs found — scraping single panel.")
#             _expand_all(page)
#             page.wait_for_timeout(500)
#             text = _extract_content_text(page)
#             sections.append(("Main", text))
#             print(f"    ✔ Main content: {len(text):,} chars")

#         browser.close()

#     # ── 4. Assemble output ────────────────────────────────────────────────
#     parts = [
#         f"SOURCE URL: {url}",
#         f"PAGE TITLE: {page_title}",
#         "=" * 60,
#         "",
#     ]

#     for tab_name, content in sections:
#         if len(sections) > 1:
#             parts.append(f"\n{'─'*40}")
#             parts.append(f"[ TAB: {tab_name} ]")
#             parts.append(f"{'─'*40}\n")
#         parts.append(content)

#     full_text = "\n".join(parts)
#     total_content_chars = sum(len(c) for _, c in sections)

#     # ── 5. Save ───────────────────────────────────────────────────────────
#     file_label = label if label else sanitize_filename(
#         page_title if page_title else url.split("/")[-1]
#     )
#     file_path = Path(WIKI_DATA_DIR) / f"{file_label}.txt"

#     with open(file_path, "w", encoding="utf-8") as f:
#         f.write(full_text)

#     print(f"\n✅ Saved: {file_path}  ({total_content_chars:,} content chars across {len(sections)} section(s))")
#     return str(file_path)


# # ── Thread wrapper (Jupyter-safe) ─────────────────────────────────────────────

# def scrape_wiki_page(url: str, label: str = None) -> str:
#     """
#     Public entry point.  Runs the Playwright scraper in a dedicated thread so
#     Jupyter's background asyncio event loop doesn't block it.
#     """
#     result = [None]
#     error  = [None]

#     def _run():
#         try:
#             result[0] = _scrape_wiki_page(url, label)
#         except Exception as e:
#             error[0] = e

#     thread = threading.Thread(target=_run, daemon=True)
#     thread.start()
#     thread.join()

#     if error[0]:
#         raise error[0]
#     return result[0]


# # ── Quick test ────────────────────────────────────────────────────────────────
# if __name__ == "__main__":
#     # Replace with any Hoyowiki Genshin entry URL
#     test_url = "https://wiki.hoyolab.com/pc/genshin/entry/4023"   # Hu Tao
#     scrape_wiki_page(test_url, label="neuvillette")

### 2a. Scrape Individual Pages

Paste any Hoyolab wiki URL below and run the cell:

In [ ]:
# --- Paste a Hoyolab wiki URL here ---
scrape_wiki_page(
    url   = "https://wiki.hoyolab.com/pc/genshin/entry/34",  # Example: Hu Tao
    label = "hu_tao"   # Optional: custom filename (no spaces)
)

### 2b. Batch-Scrape Multiple Pages

Add as many `(url, label)` pairs as you like:

In [ ]:
def find_max_entry_id(max_probe=20000, step=1000):
    """
    Probes by checking specific IDs rather than assuming
    consecutive entries — handles gaps in Hoyolab's entry IDs.
    """
    print("🔍 Probing for max entry ID...")
    
    last_valid = 1
    
    for probe_id in range(step, max_probe + step, step):
        url = f"https://sg-wiki-api.hoyolab.com/hoyowiki/genshin/wapi/entry_page?entry_page_id={probe_id}"
        try:
            resp = requests.get(url, headers={"x-rpc-language": "en-us"}, timeout=10)
            data = resp.json()
            
            if data.get("retcode") == 0 and data.get("data"):
                print(f"  ✅ Entry {probe_id} exists")
                last_valid = probe_id
            else:
                print(f"  ❌ Entry {probe_id} not found (retcode: {data.get('retcode')})")
        except Exception as e:
            print(f"  ⚠️ Error checking {probe_id}: {e}")
        
        time.sleep(0.2)  # Small delay to avoid rate limiting
    
    print(f"\n✅ Last confirmed valid entry: {last_valid}")
    print(f"   (There may be valid entries beyond this — increase max_probe if needed)")
    return last_valid

max_id = find_max_entry_id(max_probe=20000, step=1000)

In [ ]:
# Set your base URL and the range of IDs you want
base_url = "https://wiki.hoyolab.com/pc/genshin/entry/"
start_id = 2782
end_id   = 10412  # Scrape entries 1 through 1000

for entry_id in range(start_id, end_id + 1):
    url = f"{base_url}{entry_id}"
    try:
        scrape_wiki_page(url, label=f"entry_{entry_id}")
    except Exception as e:
        print(f"❌ Failed entry {entry_id}: {e}")
    time.sleep(1)  # Small delay to avoid hammering the server

### 2c. View All Saved Files

In [ ]:
saved_files = list(Path(WIKI_DATA_DIR).glob("*.txt"))

if not saved_files:
    print("⚠️  No files saved yet. Run the scraper cells above first.")
else:
    print(f"📂 {len(saved_files)} file(s) in {WIKI_DATA_DIR}:\n")
    for f in sorted(saved_files):
        size_kb = f.stat().st_size / 1024
        print(f"  • {f.name:<40} {size_kb:>6.1f} KB")

## 2d. Number of Tokens in Dataset

---
## 3. Deepseek Agent

The agent loads the saved `.txt` files and uses Deepseek to answer questions.

In [4]:
import requests
import json
from pathlib import Path

# --- Load Wiki Files ---
def load_wiki_files(filenames: list = None) -> str:
    """
    Load wiki text files into a single context string.
    - filenames=None  → loads ALL .txt files recursively
    - filenames=['Hu_Tao'] → loads only that file
    """
    data_path = Path(WIKI_DATA_DIR)

    if filenames:
        files = []
        for name in filenames:
            name = name if name.endswith(".txt") else name + ".txt"
            matches = list(data_path.rglob(name))
            if matches:
                files.append(matches[0])
            else:
                print(f"  ⚠️  File not found: {name}")
    else:
        files = sorted(data_path.rglob("*.txt"))
        files = [f for f in files if f.name != "checksums.json"]

    if not files:
        return ""

    parts = []
    for fp in files:
        if fp.exists():
            text = fp.read_text(encoding="utf-8")
            parts.append(f"\n\n{'='*60}\nFILE: {fp.name}\n{'='*60}\n{text}")
        else:
            print(f"  ⚠️  File not found: {fp}")

    return "\n".join(parts)


# --- Ollama Config ---
OLLAMA_URL   = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "deepseek-r1:8b"

# ✅ Set this to match your model's actual context window.
# DeepSeek R1 14b supports up to 128k tokens (~500k characters).
# Raise this as high as your RAM allows. Start with 32k tokens.
NUM_CTX = 32768   # tokens passed to Ollama
MAX_CHARS = NUM_CTX * 3  # ~3 chars/token is conservative for mixed content


# --- Ask Function ---
def ask(question: str, files: list = None, verbose: bool = False) -> str:
    """
    Ask DeepSeek a question using the loaded wiki files as context.

    Args:
        question : Your question about Genshin Impact
        files    : List of file labels to load (None = load all)
        verbose  : If True, show which files were loaded

    Returns:
        DeepSeek's answer as a string
    """
    wiki_context = load_wiki_files(files)

    if not wiki_context:
        return ("⚠️  No wiki files found. "
                "Please run the scraper first to save some wiki pages.")

    total_chars = len(wiki_context)

    if verbose:
        loaded = files if files else [f.name for f in Path(WIKI_DATA_DIR).rglob("*.txt")]
        print(f"📚 Loaded {len(loaded)} file(s): {', '.join(str(l) for l in loaded)}")
        print(f"📝 Total context: {total_chars:,} characters (~{total_chars//4:,} tokens)\n")

    # Warn (but don't silently truncate) if context exceeds the window
    if total_chars > MAX_CHARS:
        print(f"⚠️  Context ({total_chars:,} chars) exceeds MAX_CHARS ({MAX_CHARS:,}).")
        print(f"   Either raise NUM_CTX, target specific files, or the data will be cut off.")
        wiki_context = wiki_context[:MAX_CHARS]
    else:
        print(f"✅ Full context fits: {total_chars:,} / {MAX_CHARS:,} characters")

    prompt = f"""You are a knowledgeable Genshin Impact assistant.
Use ONLY the following wiki content to answer the question.
If the answer isn't in the content, say so clearly.

WIKI CONTENT:
{wiki_context}

QUESTION:
{question}

ANSWER:"""

    try:
        response = requests.post(
            OLLAMA_URL,
            json={
                "model":  OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False,
                "options": {
                    "temperature": 0.1,
                    "num_ctx":     NUM_CTX,   # ✅ tells Ollama to allocate a larger context window
                }
            },
            timeout=600   # longer timeout for big contexts
        )
        response.raise_for_status()
        result = response.json()
        return result.get("response", "No response received.")

    except requests.exceptions.ConnectionError:
        return ("❌ Could not connect to Ollama. "
                "Make sure Ollama is running — open a terminal and run: ollama serve")
    except Exception as e:
        return f"❌ Error: {e}"


# --- Test Connection ---
def test_connection():
    try:
        resp = requests.get("http://localhost:11434/api/tags", timeout=5)
        models = [m["name"] for m in resp.json().get("models", [])]
        if OLLAMA_MODEL in models:
            print(f"✅ Ollama running — {OLLAMA_MODEL} is available")
        else:
            print(f"⚠️  Ollama running but {OLLAMA_MODEL} not found")
            print(f"   Available models: {models}")
            print(f"   Run in terminal: ollama pull deepseek-r1:14b")
    except Exception:
        print("❌ Ollama is not running — open a terminal and run: ollama serve")

test_connection()

✅ Ollama running — deepseek-r1:8b is available


In [6]:
def count_tokens(files: list = None, question: str = None) -> dict:
    """
    Estimates token count for your input before sending to DeepSeek.
    
    Args:
        files    : List of file labels to load (None = load all)
        question : Optional question to include in the count
    
    Returns:
        Dict with character and token counts
    """
    wiki_context = load_wiki_files(files)
    
    if not wiki_context:
        print("⚠️  No wiki files found.")
        return {}
    
    # Include question in count if provided
    full_input = wiki_context
    if question:
        full_input += f"\n\nQUESTION:\n{question}"
    
    # Estimate tokens — rule of thumb is ~4 characters per token for English
    char_count    = len(full_input)
    token_estimate = char_count // 4
    
    # DeepSeek R1 7B context limits
    MODEL_LIMIT = 4096
    SAFE_LIMIT  = 3500  # Leave room for the answer
    
    status = "✅ Safe" if token_estimate <= SAFE_LIMIT else "⚠️  Too large"
    
    print(f"📄 Characters:       {char_count:>10,}")
    print(f"🔢 Estimated tokens: {token_estimate:>10,}")
    print(f"📏 Model limit:      {MODEL_LIMIT:>10,}")
    print(f"📏 Safe limit:       {SAFE_LIMIT:>10,}")
    print(f"📊 Usage:            {token_estimate:>10,} / {MODEL_LIMIT:,}  ({token_estimate/MODEL_LIMIT*100:.1f}%)")
    print(f"🚦 Status:           {status}")
    
    if token_estimate > SAFE_LIMIT:
        recommended = (SAFE_LIMIT * 4)
        print(f"\n💡 Recommendation: truncate context to {recommended:,} characters")
        print(f"   In ask(), set MAX_CHARS = {recommended}")
    
    return {
        "characters":    char_count,
        "tokens":        token_estimate,
        "model_limit":   MODEL_LIMIT,
        "safe_limit":    SAFE_LIMIT,
        "fits":          token_estimate <= SAFE_LIMIT
    }

count_tokens()  # No arguments = loads everything

📄 Characters:       16,721,126
🔢 Estimated tokens:  4,180,281
📏 Model limit:           4,096
📏 Safe limit:            3,500
📊 Usage:             4,180,281 / 4,096  (102057.6%)
🚦 Status:           ⚠️  Too large

💡 Recommendation: truncate context to 14,000 characters
   In ask(), set MAX_CHARS = 14000


{'characters': 16721126,
 'tokens': 4180281,
 'model_limit': 4096,
 'safe_limit': 3500,
 'fits': False}

In [5]:
import requests
import chromadb
from sentence_transformers import SentenceTransformer
from pathlib import Path

# --- Embedding Model & ChromaDB Setup ---
embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")
client     = chromadb.PersistentClient(path="./chroma_db")
collection = client.get_or_create_collection("genshin_wiki")


# --- Chunking with Overlap ---
def chunk_text(text: str, size: int = 1000, overlap: int = 200) -> list:
    """
    Split text into overlapping chunks so context isn't lost at boundaries.
    e.g. size=1000, overlap=200 means each chunk shares 200 chars with the next.
    """
    chunks = []
    i = 0
    while i < len(text):
        chunks.append(text[i:i + size])
        i += size - overlap
    return chunks


# --- Index Files (run once) ---
def index_files():
    """
    Embed and store all wiki files in ChromaDB.
    Run once after scraping, or again if you add new files.
    """
    files = sorted(Path(WIKI_DATA_DIR).rglob("*.txt"))
    files = [f for f in files if f.name != "checksums.json"]

    if not files:
        print("⚠️  No wiki files found. Run the scraper first.")
        return

    print(f"📥 Indexing {len(files)} file(s)...")

    for fp in files:
        text     = fp.read_text(encoding="utf-8")
        chunks   = chunk_text(text, size=1000, overlap=200)
        embeddings = embedder.encode(chunks).tolist()

        collection.upsert(
            documents  = chunks,
            embeddings = embeddings,
            ids        = [f"{fp.stem}_{i}" for i in range(len(chunks))],
            metadatas  = [{"source": fp.stem.lower()} for _ in chunks]  # ← filename stored for filtering
        )
        print(f"  ✅ Indexed: {fp.name}  ({len(chunks)} chunks)")

    print("\n✅ Indexing complete.")


# --- Ask with RAG ---
def ask_rag(question: str, top_k: int = 20, verbose: bool = False) -> str:
    """
    Retrieve relevant chunks from ChromaDB, then ask DeepSeek.

    Args:
        question : Your question about Genshin Impact
        top_k    : How many chunks to retrieve (raise if answers feel incomplete)
        verbose  : If True, show which chunks were retrieved

    Returns:
        DeepSeek's answer as a string
    """

    # --- Step 1: Check if any word in the question matches a filename ---
    question_words = question.lower().replace("'", "").replace("?", "").split()
    all_files      = [f.stem.lower() for f in Path(WIKI_DATA_DIR).rglob("*.txt")]
    file_filter    = None

    for word in question_words:
        if word in all_files:
            file_filter = word
            break

    # --- Step 2: Query ChromaDB ---
    query_embedding = embedder.encode([question]).tolist()

    if file_filter:
        print(f"📌 Filtering to file: {file_filter}.txt")
        results = collection.query(
            query_embeddings = query_embedding,
            n_results        = top_k,
            where            = {"source": file_filter}
        )
    else:
        results = collection.query(
            query_embeddings = query_embedding,
            n_results        = top_k
        )

    # --- Step 3: Build context from retrieved chunks ---
    retrieved_chunks = results["documents"][0]

    if not retrieved_chunks:
        return "⚠️  No relevant wiki content found. Try rephrasing or check that the file is indexed."

    if verbose:
        sources = [m["source"] for m in results["metadatas"][0]]
        print(f"📚 Retrieved {len(retrieved_chunks)} chunk(s) from: {set(sources)}\n")
        for i, chunk in enumerate(retrieved_chunks, 1):
            print(f"  [{i}] {chunk[:80]}...")

    context = "\n\n".join(retrieved_chunks)

    # --- Step 4: Send to DeepSeek ---
    prompt = f"""You are a knowledgeable Genshin Impact assistant.
Use ONLY the following wiki content to answer the question.
If the answer isn't in the content, say so clearly — do not guess.

WIKI CONTENT:
{context}

QUESTION:
{question}

ANSWER:"""

    try:
        resp = requests.post(
            OLLAMA_URL,
            json={
                "model":  OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False,
                "options": {
                    "temperature": 0.1,
                    "num_ctx":     NUM_CTX,
                }
            }
            # timeout=300
        )
        resp.raise_for_status()
        return resp.json().get("response", "No response received.")

    except requests.exceptions.ConnectionError:
        return "❌ Could not connect to Ollama. Make sure it's running: ollama serve"
    except Exception as e:
        return f"❌ Error: {e}"


# --- Run indexing, then ask ---
# index_files()
# answer = ask_rag("What is the best artifact set for Neuvillette's dps build", verbose=True)
# print(answer)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Keyword Extractor

In [6]:
import re

# Common words to ignore — add more as needed
STOPWORDS = {
    "what", "is", "the", "a", "an", "for", "of", "in", "on", "to",
    "and", "or", "how", "does", "do", "best", "are", "with", "my",
    "good", "give", "me", "can", "which", "that", "it", "be", "use"
}

def extract_keywords(question: str) -> list[str]:
    """Pull out meaningful words from a question."""
    words = question.lower()
    words = re.sub(r"[^\w\s]", "", words)  # remove punctuation
    words = words.split()
    keywords = [w for w in words if w not in STOPWORDS and len(w) > 2]
    return keywords

# Test it
extract_keywords("What is the best artifact for a dps Neuvillette?")
# → ['artifact', 'dps', 'neuvillette']

['artifact', 'dps', 'neuvillette']

Keyword Reiltered REtrieval

In [7]:
def ask_rag(question, top_k=48, verbose=False):
    keywords = extract_keywords(question)
    
    if verbose:
        print(f"🔑 Keywords extracted: {keywords}")

    query_embedding = embedder.encode([question]).tolist()

    # --- Try keyword-filtered search first ---
    # ChromaDB's $contains only supports one keyword at a time,
    # so we query once per keyword and merge results
    seen_ids = set()
    retrieved_chunks = []
    retrieved_meta   = []

    for keyword in keywords:
        results = collection.query(
            query_embeddings=query_embedding,
            n_results=top_k,
            where_document={"$contains": keyword}
        )
        for doc, meta, rid in zip(
            results["documents"][0],
            results["metadatas"][0],
            results["ids"][0]
        ):
            if rid not in seen_ids:
                seen_ids.add(rid)
                retrieved_chunks.append(doc)
                retrieved_meta.append(meta)

    # --- Fall back to plain vector search if nothing found ---
    if not retrieved_chunks:
        if verbose:
            print("⚠️  No keyword matches — falling back to vector search")
        results = collection.query(
            query_embeddings=query_embedding,
            n_results=top_k
        )
        retrieved_chunks = results["documents"][0]
        retrieved_meta   = results["metadatas"][0]

    if verbose:
        sources = {m["source"] for m in retrieved_meta}
        print(f"📚 Retrieved {len(retrieved_chunks)} chunk(s) from: {sources}\n")
        for i, chunk in enumerate(retrieved_chunks, 1):
            print(f"  [{i}] {chunk[:80]}...")

    context = "\n\n".join(retrieved_chunks)

    # --- Send to DeepSeek (unchanged) ---
    prompt = f"""You are a knowledgeable Genshin Impact assistant. You have been given wiki content to answer the user's question.

You will encounter two types of questions:

1. FACTUAL questions — These have a direct answer in the wiki content.
   Examples: "What element does Neuvillette use?", "What is Hu Tao's constellation name?"
   → Answer directly and concisely from the content. Do not add opinions.

2. ANALYTICAL questions — These require you to reason over the content to form a recommendation or conclusion.
   Examples: "What is the best artifact set for Neuvillette?", "Should I use Neuvillette or Furina as my main DPS?"
   → Think step by step. Weigh the options in the content, explain your reasoning, then give a clear recommendation.

RULES:
- Use ONLY the wiki content below. Do not use outside knowledge.
- If the answer is not in the content, say so clearly — do not guess.
- If the question is analytical, always end with a clear final recommendation.

WIKI CONTENT:
{context}

QUESTION:
{question}

ANSWER:"""

    try:
        resp = requests.post(
            OLLAMA_URL,
            json={
                "model":  OLLAMA_MODEL,
                "prompt": prompt,
                "stream": False,
                "options": {"temperature": 0.1, "num_ctx": NUM_CTX}
            }
        )
        resp.raise_for_status()
        return resp.json().get("response", "No response received.")
    except requests.exceptions.ConnectionError:
        return "❌ Could not connect to Ollama. Make sure it's running: ollama serve"
    except Exception as e:
        return f"❌ Error: {e}"

---
## 4. Ask Questions!

### Simple query (searches all saved files)

In [ ]:
answer = ask_rag("What element does Hu Tao use and what is her role in a team?")
print(answer)

In [8]:
answer = ask_rag("Who is Neuvillette, and what does he do?", verbose=True)
print(answer)

🔑 Keywords extracted: ['who', 'neuvillette']
📚 Retrieved 48 chunk(s) from: {'entry_4599', 'entry_5146', 'entry_6274', 'entry_7473', 'entry_4598', 'entry_4601', 'entry_3979', 'entry_4808', 'entry_5144', 'entry_4924', 'entry_5145', 'entry_4916', 'entry_4781', 'entry_4778', 'entry_4723', 'entry_9302', 'entry_5222', 'entry_4576', 'entry_4603', 'entry_4600', 'entry_5142', 'entry_4722', 'entry_5223'}

  [1]  to Snezhnaya to recover from his wounds.
When Traveler and Paimon finished the ...
  [2] of his henchmen, fled to Poisson, taking hostage both the city itself and the me...
  [3] a fiery performance, they accidentally activated the Palais's fire-fighting mech...
  [4] came one of the first Melusines who, with the help of the young Iudex of Fontain...
  [5] you. Neuvillette was not opposed to such a manner, but neither did he approve of...
  [6] Archon. When Neuvillette relayed Focalors' words to her, she merely replied that...
  [7] lusines and told them to return to their village. Neuvi

In [9]:
answer = ask_rag("What is the best artifact set for Neuvillette's dps build?")
print(answer)

Based on the wiki content provided:

**Best Artifact Set Recommendation:**

For Neuvillette's main DPS role, focusing on **Elemental Burst (Hydro) DMG**, the optimal artifact set combination is:
- **2-Piece:** *The Exile*  
  - Increases Energy Recharge by +20%. This helps maintain energy for her Elemental Burst.
- **4-Piece:** *Emblem of Severed Fate*  
  - Increases Elemental Burst DMG by +25% based on Energy Recharge, with a maximum bonus of +75%.

**Reasoning:**
This combination leverages the synergy between high Energy Recharge (from The Exile) and burst damage scaling (from Emblem of Severed Fate). It is efficient for maximizing DPS in Hydro-focused teams.

**Final Recommendation:**  
Use **The Exile 2-piece + Emblem of Severed Fate 4-piece** to optimize Neuvillette's Elemental Burst DMG through Energy Recharge.


In [ ]:
all_files = [f.stem.lower() for f in Path(WIKI_DATA_DIR).rglob("*.txt")]
print(all_files)

### Target specific files (faster, more focused)

In [ ]:
answer = ask(
    question = "What are Hu Tao's best weapons?",
    files    = ["hu_tao"],   # Only load hu_tao.txt
    verbose  = True
)
print(answer)

### Interactive chat loop

Keep asking questions until you type `quit`:

In [ ]:
print("🤖 Genshin Wiki Agent — type 'quit' to exit\n")
print(f"📂 Loaded files: {[f.name for f in Path(WIKI_DATA_DIR).glob('*.txt')]}\n")

while True:
    question = input("You: ").strip()
    if not question:
        continue
    if question.lower() in ("quit", "exit", "q"):
        print("👋 Bye!")
        break
    
    print("\n🌸 Gemini:")
    answer = ask_rag(question, top_k=5)
    print(answer)
    print("\n" + "-"*50 + "\n")

---
## 5. Utilities

### Preview a saved file

In [ ]:
def preview_file(label: str, lines: int = 50):
    fp = Path(WIKI_DATA_DIR) / f"{label}.txt"
    if not fp.exists():
        print(f"File not found: {fp}")
        return
    content = fp.read_text(encoding="utf-8").splitlines()
    print(f"\n--- {fp.name} (first {lines} lines) ---\n")
    print("\n".join(content[:lines]))
    if len(content) > lines:
        print(f"\n... ({len(content) - lines} more lines)")

preview_file("hu_tao")

### Delete a saved file

In [ ]:
def delete_file(label: str):
    fp = Path(WIKI_DATA_DIR) / f"{label}.txt"
    if fp.exists():
        fp.unlink()
        print(f"🗑️  Deleted: {fp.name}")
    else:
        print(f"File not found: {fp}")

# delete_file("hu_tao")   # Uncomment to delete

---
## Quick Reference

| Task | Code |
|------|------|
| Scrape a page | `scrape_wiki_page(url, label)` |
| List saved files | `list(Path(WIKI_DATA_DIR).glob('*.txt'))` |
| Ask using all files | `ask('your question')` |
| Ask using specific files | `ask('question', files=['hu_tao', 'amber'])` |
| Preview a file | `preview_file('hu_tao')` |
| Delete a file | `delete_file('hu_tao')` |

**Good Hoyolab wiki pages to scrape:**
- Characters: `https://wiki.hoyolab.com/pc/genshin/entry/{id}`
- Browse the wiki at: https://wiki.hoyolab.com/pc/genshin/home